In [21]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# 2. Импорты
import time, numpy as np, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn import L1Loss, MSELoss, HuberLoss
from torch.cuda.amp import autocast, GradScaler

# 3. Устройство и AMP scaler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
scaler = GradScaler()

# 4. Метрики
def smape(y_true, y_pred):
    denom = (y_true.abs() + y_pred.abs())/2
    mask = denom > 0
    return 100*((y_pred[mask]-y_true[mask]).abs()/denom[mask]).mean().item()
def r2_torch(y_true, y_pred):
    ss_res = ((y_true-y_pred)**2).sum()
    ss_tot = ((y_true-y_true.mean())**2).sum()
    return (1 - ss_res/ss_tot).item()

# 5. Загрузка данных + Dataset
pressure = np.loadtxt("pressure.txt")[:,1:]
velocity = np.loadtxt("velocity.txt")[:,1:]
X = torch.tensor(pressure, dtype=torch.float32)
y = torch.tensor(velocity, dtype=torch.float32)

class ShockTubeDS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

ds = ShockTubeDS(X, y)
n = len(ds)
n_train = int(0.8*n)
train_ds, test_ds = random_split(ds, [n_train, n-n_train], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16)

# 6. Определение MLP
class MLP(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(seq_len, 128), nn.ReLU(),
            nn.Linear(128, 64),      nn.ReLU(),
            nn.Linear(64, seq_len)
        )
    def forward(self, x): return self.net(x)

# 7. Train/Eval функции
def train_epoch(model, loader, crit, opt, scaler):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        with autocast():
            loss = crit(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total += loss.item() * xb.size(0)
    return total/len(loader.dataset)

def eval_epoch(model, loader, crit):
    model.eval()
    total=0
    with torch.no_grad(), autocast():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            total += crit(model(xb), yb).item() * xb.size(0)
    return total/len(loader.dataset)

# 8. Обучение MLP
seq_len = X.shape[1]
model = MLP(seq_len).to(device)
opt   = optim.Adam(model.parameters(), lr=1e-3)
crit  = HuberLoss(delta=15.0).to(device)
mae_crit  = L1Loss().to(device)
mse_crit  = MSELoss().to(device)

for ep in range(1, 21):
    t0 = time.time()
    tr = train_epoch(model, train_loader, crit, opt, scaler)
    vl = eval_epoch (model, test_loader,  crit)
    dt = time.time()-t0

    # собираем предсказания для метрик
    all_pred, all_true = [], []
    with torch.no_grad(), autocast():
        for xb, yb in test_loader:
            xb = xb.to(device)
            all_pred.append(model(xb).cpu())
            all_true.append(yb)
    pred = torch.cat(all_pred)
    true = torch.cat(all_true)

    mae = mae_crit(pred, true).item()
    rmse= torch.sqrt(mse_crit(pred,true)).item()
    sm = smape(true,pred)
    r2= r2_torch(true,pred)

    print(f"Epoch {ep:02d} | TrainL={tr:.4f} ValL={vl:.4f} | "
          f"MAE={mae:.4f} RMSE={rmse:.4f} SMAPE={sm:.2f}% R2={r2:.4f} | {dt:.2f}s")

# 9. Удаление модели и очистка
del model, opt, crit, scaler
torch.cuda.empty_cache()


Device: cuda


<ipython-input-21-13ec1f2bfff2>:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-21-13ec1f2bfff2>:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
<ipython-input-21-13ec1f2bfff2>:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
<ipython-input-21-13ec1f2bfff2>:96: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Epoch 01 | TrainL=0.0042 ValL=0.0012 | MAE=0.0312 RMSE=0.0483 SMAPE=135.43% R2=-0.0182 | 0.16s
Epoch 02 | TrainL=0.0010 ValL=0.0010 | MAE=0.0267 RMSE=0.0437 SMAPE=126.61% R2=0.1694 | 0.15s
Epoch 03 | TrainL=0.0009 ValL=0.0009 | MAE=0.0266 RMSE=0.0435 SMAPE=127.48% R2=0.1737 | 0.15s
Epoch 04 | TrainL=0.0009 ValL=0.0010 | MAE=0.0268 RMSE=0.0437 SMAPE=128.76% R2=0.1679 | 0.16s
Epoch 05 | TrainL=0.0009 ValL=0.0009 | MAE=0.0265 RMSE=0.0435 SMAPE=127.14% R2=0.1736 | 0.15s
Epoch 06 | TrainL=0.0009 ValL=0.0009 | MAE=0.0266 RMSE=0.0435 SMAPE=128.83% R2=0.1752 | 0.15s
Epoch 07 | TrainL=0.0009 ValL=0.0010 | MAE=0.0268 RMSE=0.0437 SMAPE=126.22% R2=0.1683 | 0.15s
Epoch 08 | TrainL=0.0009 ValL=0.0010 | MAE=0.0268 RMSE=0.0436 SMAPE=129.20% R2=0.1704 | 0.15s
Epoch 09 | TrainL=0.0009 ValL=0.0009 | MAE=0.0265 RMSE=0.0435 SMAPE=127.42% R2=0.1755 | 0.15s
Epoch 10 | TrainL=0.0009 ValL=0.0009 | MAE=0.0266 RMSE=0.0435 SMAPE=125.96% R2=0.1736 | 0.15s
Epoch 11 | TrainL=0.0009 ValL=0.0009 | MAE=0.0266 RMSE=0.04

In [31]:
# Cell: Training 1D-CNN on GPU

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import time
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn import MSELoss, L1Loss
from torch.cuda.amp import autocast, GradScaler

# 0. Device + AMP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()
print("Device:", device)

# 1. Metrics (no SMAPE)
def r2_torch(y, yhat):
    ss_res = ((y - yhat)**2).sum()
    ss_tot = ((y - y.mean())**2).sum()
    return (1 - ss_res/ss_tot).item()

# 2. Load, downsample and normalize
pressure = np.loadtxt("pressure.txt")[:,1::5]  # every 5th time step
velocity = np.loadtxt("velocity.txt")[:,1::5]
# compute stats on train portion only?
P_mean, P_std = pressure.mean(), pressure.std()
U_mean, U_std = velocity.mean(), velocity.std()
P_n = (pressure - P_mean)/P_std
U_n = (velocity - U_mean)/U_std

X = torch.tensor(P_n, dtype=torch.float32)
y = torch.tensor(U_n, dtype=torch.float32)

# 3. Dataset+Loader
class ShockTubeDS(Dataset):
    def __init__(self,X,y): self.X,self.y = X,y
    def __len__(self):   return len(self.X)
    def __getitem__(self,i): return self.X[i], self.y[i]

ds = ShockTubeDS(X,y)
n = len(ds)
train_ds, test_ds = random_split(ds, [int(0.8*n), n-int(0.8*n)],
                                 generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32)

# 4. 1D-CNN model
class CNN1D(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(16,32, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=5, padding=2)
        )
    def forward(self, x):
        x = x.unsqueeze(1)         # (B,1,seq)
        out = self.net(x)          # (B,1,seq)
        return out.squeeze(1)      # (B,seq)

# 5. Train/val functions
def train_epoch(model, loader, crit, opt, scaler):
    model.train()
    tot=0
    for xb,yb in loader:
        xb,yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        with autocast():
            loss = crit(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        tot += loss.item()*xb.size(0)
    return tot/len(loader.dataset)

def eval_epoch(model, loader, crit):
    model.eval()
    tot=0
    with torch.no_grad(), autocast():
        for xb,yb in loader:
            xb,yb = xb.to(device), yb.to(device)
            tot += crit(model(xb), yb).item()*xb.size(0)
    return tot/len(loader.dataset)

# 6. Setup
seq_len = X.shape[1]
model     = CNN1D(seq_len).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = MSELoss().to(device)
mae_crit  = L1Loss().to(device)

# 7. Training loop
epochs = 200
for ep in range(1, epochs+1):
    t0 = time.time()
    tr = train_epoch(model, train_loader, criterion, optimizer, scaler)
    vl = eval_epoch (model, test_loader,  criterion)
    dt = time.time() - t0

    # predictions + denorm + compute MAE,R2
    all_p, all_t = [], []
    with torch.no_grad(), autocast():
        for xb,yb in test_loader:
            xb = xb.to(device)
            pred_n = model(xb).cpu()
            all_p.append(pred_n*U_std + U_mean)
            all_t.append(yb*U_std + U_mean)
    pred = torch.cat(all_p)
    true = torch.cat(all_t)

    mae = mae_crit(pred, true).item()
    r2  = r2_torch(true, pred)

    print(f"Epoch {ep:02d} │ TrainMSE={tr:.4f} ValMSE={vl:.4f} "
          f"MAE={mae:.4f} R2={r2:.4f} │ {dt:.2f}s")

# 8. Cleanup
del model, optimizer, criterion, scaler
torch.cuda.empty_cache()


<ipython-input-31-5cfdcbc49f09>:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Device: cuda


<ipython-input-31-5cfdcbc49f09>:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
<ipython-input-31-5cfdcbc49f09>:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
<ipython-input-31-5cfdcbc49f09>:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Epoch 01 │ TrainMSE=0.9876 ValMSE=0.9952 MAE=0.0287 R2=0.0148 │ 0.09s
Epoch 02 │ TrainMSE=0.9747 ValMSE=0.9879 MAE=0.0287 R2=0.0220 │ 0.08s
Epoch 03 │ TrainMSE=0.9648 ValMSE=0.9753 MAE=0.0286 R2=0.0345 │ 0.08s
Epoch 04 │ TrainMSE=0.9534 ValMSE=0.9656 MAE=0.0286 R2=0.0441 │ 0.08s
Epoch 05 │ TrainMSE=0.9426 ValMSE=0.9580 MAE=0.0286 R2=0.0517 │ 0.08s
Epoch 06 │ TrainMSE=0.9323 ValMSE=0.9450 MAE=0.0285 R2=0.0645 │ 0.08s
Epoch 07 │ TrainMSE=0.9222 ValMSE=0.9353 MAE=0.0284 R2=0.0741 │ 0.08s
Epoch 08 │ TrainMSE=0.9108 ValMSE=0.9232 MAE=0.0283 R2=0.0861 │ 0.08s
Epoch 09 │ TrainMSE=0.9009 ValMSE=0.9164 MAE=0.0282 R2=0.0928 │ 0.08s
Epoch 10 │ TrainMSE=0.8923 ValMSE=0.9081 MAE=0.0282 R2=0.1010 │ 0.08s
Epoch 11 │ TrainMSE=0.8855 ValMSE=0.9001 MAE=0.0281 R2=0.1089 │ 0.08s
Epoch 12 │ TrainMSE=0.8783 ValMSE=0.8960 MAE=0.0280 R2=0.1130 │ 0.08s
Epoch 13 │ TrainMSE=0.8725 ValMSE=0.8945 MAE=0.0280 R2=0.1145 │ 0.08s
Epoch 14 │ TrainMSE=0.8688 ValMSE=0.8860 MAE=0.0280 R2=0.1229 │ 0.08s
Epoch 15 │ TrainMSE=

In [32]:
# Cell: Training a DeepONet model on GPU

import os
# Reduce CUDA fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import time
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import IterableDataset, DataLoader
from torch.nn import MSELoss, L1Loss
from torch.cuda.amp import autocast, GradScaler

# 0. Device and AMP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
scaler = GradScaler()

# 1. Load and normalize data
data_p = np.loadtxt("pressure.txt")
data_u = np.loadtxt("velocity.txt")
x_coords = data_p[:, 0].astype(np.float32)  # shape (Nx,)
P = data_p[:, 1:].astype(np.float32)        # shape (Nx, Nt)
U = data_u[:, 1:].astype(np.float32)

P_mean, P_std = P.mean(), P.std()
U_mean, U_std = U.mean(), U.std()
P_n = (P - P_mean) / P_std
U_n = (U - U_mean) / U_std

Nx, Nt = P_n.shape

# 2. IterableDataset sampling random (branch, trunk, target) pairs
class RandomDeepONetDataset(IterableDataset):
    def __init__(self, P, U, x, samples_per_epoch):
        self.P = torch.tensor(P, dtype=torch.float32)
        self.U = torch.tensor(U, dtype=torch.float32)
        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # (Nx,1)
        self.Nx = self.P.shape[0]
        self.Nt = self.P.shape[1]
        self.samples = samples_per_epoch

    def __iter__(self):
        for _ in range(self.samples):
            # random time index and spatial index
            n = torch.randint(self.Nt, (1,)).item()
            i = torch.randint(self.Nx, (1,)).item()
            branch = self.P[:, n]        # (Nx,)
            trunk  = self.x[i]           # (1,)
            target = self.U[i, n]        # scalar
            yield branch, trunk, target

# 3. Create loaders
train_samples = 200_000
test_samples  = 50_000

train_ds = RandomDeepONetDataset(P_n, U_n, x_coords, train_samples)
test_ds  = RandomDeepONetDataset(P_n, U_n, x_coords, test_samples)

train_loader = DataLoader(train_ds, batch_size=512)
test_loader  = DataLoader(test_ds,  batch_size=512)

# 4. Define DeepONet model
class DeepONet(nn.Module):
    def __init__(self, branch_dim, trunk_dim, p_dim=64):
        super().__init__()
        # branch: maps discretized function -> p_dim features
        self.branch_net = nn.Sequential(
            nn.Linear(branch_dim, 128),
            nn.ReLU(),
            nn.Linear(128, p_dim)
        )
        # trunk: maps coordinate -> p_dim features
        self.trunk_net = nn.Sequential(
            nn.Linear(trunk_dim, 128),
            nn.ReLU(),
            nn.Linear(128, p_dim)
        )

    def forward(self, branch, trunk):
        # branch: (B, Nx), trunk: (B, 1)
        b = self.branch_net(branch)  # (B, p_dim)
        t = self.trunk_net(trunk)    # (B, p_dim)
        return (b * t).sum(dim=1)    # dot product -> (B,)

# 5. Initialize model, criterion, optimizer
model = DeepONet(branch_dim=Nx, trunk_dim=1, p_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = MSELoss().to(device)
mae_crit  = L1Loss().to(device)

# 6. Training loop
epochs = 20
for epoch in range(1, epochs+1):
    t0 = time.time()
    # train
    model.train()
    total_loss = 0.0
    for branch, trunk, target in train_loader:
        branch, trunk, target = branch.to(device), trunk.to(device), target.to(device)
        optimizer.zero_grad()
        with autocast():
            pred = model(branch, trunk)
            loss = criterion(pred, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * branch.size(0)
    train_loss = total_loss / train_samples

    # validate
    model.eval()
    total_loss = 0.0
    all_pred, all_true = [], []
    with torch.no_grad(), autocast():
        for branch, trunk, target in test_loader:
            branch, trunk, target = branch.to(device), trunk.to(device), target.to(device)
            pred_n = model(branch, trunk)
            total_loss += criterion(pred_n, target).item() * branch.size(0)
            all_pred.append(pred_n.cpu())
            all_true.append(target.cpu())
    val_loss = total_loss / test_samples

    # compute MAE and R2 on test batch
    pred_full = torch.cat(all_pred)
    true_full = torch.cat(all_true)
    mae = mae_crit(pred_full, true_full).item()
    ss_res = ((true_full - pred_full)**2).sum()
    ss_tot = ((true_full - true_full.mean())**2).sum()
    r2 = (1 - ss_res/ss_tot).item()

    dt = time.time() - t0
    print(f"Epoch {epoch:02d} │ TrainMSE={train_loss:.4f} ValMSE={val_loss:.4f} "
          f"MAE={mae:.4f} R2={r2:.4f} │ {dt:.2f}s")

# 7. Cleanup
del model, optimizer, criterion, scaler
torch.cuda.empty_cache()



Device: cuda


<ipython-input-32-935324aa666c>:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-32-935324aa666c>:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
<ipython-input-32-935324aa666c>:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Epoch 01 │ TrainMSE=1.8946 ValMSE=0.7566 MAE=0.4310 R2=0.2391 │ 10.81s
Epoch 02 │ TrainMSE=0.7529 ValMSE=0.7080 MAE=0.4243 R2=0.2664 │ 10.76s
Epoch 03 │ TrainMSE=0.7114 ValMSE=0.8131 MAE=0.4628 R2=0.2190 │ 10.76s
Epoch 04 │ TrainMSE=0.7457 ValMSE=0.6655 MAE=0.4100 R2=0.3328 │ 10.85s
Epoch 05 │ TrainMSE=0.7566 ValMSE=0.6585 MAE=0.4009 R2=0.3545 │ 10.64s
Epoch 06 │ TrainMSE=0.6966 ValMSE=0.6910 MAE=0.4176 R2=0.3159 │ 10.53s
Epoch 07 │ TrainMSE=0.6869 ValMSE=0.6861 MAE=0.4180 R2=0.3267 │ 10.70s
Epoch 08 │ TrainMSE=0.6939 ValMSE=0.6238 MAE=0.4102 R2=0.3601 │ 10.58s
Epoch 09 │ TrainMSE=0.6688 ValMSE=0.5830 MAE=0.3873 R2=0.4051 │ 10.69s
Epoch 10 │ TrainMSE=0.6203 ValMSE=0.7913 MAE=0.4045 R2=0.2185 │ 10.57s
Epoch 11 │ TrainMSE=0.6069 ValMSE=0.7485 MAE=0.4335 R2=0.2281 │ 10.55s
Epoch 12 │ TrainMSE=0.5731 ValMSE=0.5482 MAE=0.3667 R2=0.4514 │ 10.53s
Epoch 13 │ TrainMSE=0.5806 ValMSE=0.5180 MAE=0.3719 R2=0.4856 │ 10.63s
Epoch 14 │ TrainMSE=0.5659 ValMSE=0.4638 MAE=0.3434 R2=0.5237 │ 10.66s
Epoch 